# Mini-SGLang 第一次推理路径闯关

这份教程只关注一件事：**一个 prompt 怎样变成模型输出**。

我们会用一条请求、单张 GPU、`naive` KV cache、`page_size=1`，并关闭 CUDA Graph 与 overlap scheduling。每一关只看当前需要的对象；答对当前问题，才能解锁下一关。

> 这是 offline 路径：它跳过 API Server、Tokenizer worker 和 ZMQ，把注意力集中在 Scheduler → Engine → Decode。在线多进程部分留到第二次学习。


## 规则

- 按顺序执行 code cell，不要跳过。
- 每个关卡的最后一个 `input()` 是门禁。
- 答错会抛出 `RuntimeError`；重新运行本关并答对后，才能继续。
- 第一遍不需要读懂 kernel 内部，只要理解每个对象在系统中的职责。
- 看到 `(Pdb)` 以外的普通输出是正常的；本教程不使用 PDB。


In [5]:
# 准备环境。不要在构造 LLM 前调用任何 torch.cuda 查询。
import json
import os
import sys
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("MINISGL_DISABLE_OVERLAP_SCHEDULING", "1")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ["PATH"] = "/root/miniconda3/bin:" + os.environ.get("PATH", "")

ROOT = Path("/root/workspace/mini-sglang")
sys.path.insert(0, str(ROOT / "python"))

import torch
from minisgl.core import SamplingParams
from minisgl.llm import LLM
from minisgl.message import UserMsg
from minisgl.llm.llm import RequestStatus

MODEL = "/root/workspace/models/Qwen2.5-0.5B-Instruct"
PROMPT = "Explain KV cache in one short sentence."
MAX_TOKENS = 4

print("torch:", torch.__version__)
print("CUDA check deferred until LLM constructs Engine.")
print("model:", MODEL)

# LLM 初始化会加载模型、分配 KV cache，并让 Engine 成为第一个 CUDA 初始化者。
llm = LLM(
    model_path=MODEL,
    dtype=torch.bfloat16,
    max_running_req=1,
    max_extend_tokens=256,
    page_size=1,
    cache_type="naive",
    cuda_graph_max_bs=0,
)

lesson = {"unlocked": 0, "uid": 0, "prompt": PROMPT}
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("LLM ready")

def gate(level, question, accepted):
    if lesson["unlocked"] != level - 1:
        raise RuntimeError(f"请先完成第 {lesson['unlocked'] + 1} 关，再进入第 {level} 关。")
    raw_answers = os.getenv("MINISGL_NOTEBOOK_ANSWERS")
    if raw_answers:
        answers = json.loads(raw_answers)
        answer = str(answers[level - 1])
    else:
        answer = input(question + "\n> ").strip().lower()
    if not any(word in answer for word in accepted):
        raise RuntimeError("回答还不够准确：请根据本关展示的状态重新思考，再运行本关。")
    lesson["unlocked"] = level
    print(f"第 {level} 关通过。")

def req_state(req):
    return {
        "uid": req.uid,
        "cached_len": req.cached_len,
        "device_len": req.device_len,
        "remain_len": req.remain_len,
        "can_decode": req.can_decode,
    }


torch: 2.7.0+cu128
CUDA check deferred until LLM constructs Engine.
model: /root/workspace/models/Qwen2.5-0.5B-Instruct


AssertionError: 

## 第 1 关：请求对象

先只做 tokenizer 的工作。`LLM._tokenize_one()` 把字符串变成 CPU 上的 token IDs；这时还没有进入 Scheduler，也没有执行模型。

**问题：字符串 prompt 在这一关变成了什么？**


In [2]:
params = SamplingParams(temperature=0.0, max_tokens=MAX_TOKENS)
input_ids = llm._tokenize_one(PROMPT)
lesson.update({"input_ids": input_ids, "params": params})
llm.status_map = {0: RequestStatus(uid=0, input_ids=input_ids.tolist(), output_ids=[])}
print("prompt:", PROMPT)
print("input_ids device:", input_ids.device)
print("input token IDs:", input_ids.tolist())
print("number of input tokens:", len(input_ids))
print("sampling params:", params)

gate(1, "第 1 关：输出中哪一行是 prompt 变换后的数字序列？请回答 token IDs：", ["token", "词元", "ids"])


prompt: Explain KV cache in one short sentence.
input_ids device: cpu
number of input tokens: 9
sampling params: SamplingParams(temperature=0.0, top_k=-1, top_p=1.0, ignore_eos=False, max_tokens=4)


第 1 关：prompt 现在变成了什么？请回答 token / token ids / 字符串之一：
>  token ids


第 1 关通过。


## 第 2 关：请求进入 Scheduler

offline 模式用 `UserMsg` 模拟 tokenizer worker 发给 Scheduler 的消息。Scheduler 收到消息后，会把它包装成 `PendingReq` 放入 `pending_list`。

**问题：此时请求已经在 GPU 上运行了吗？**
请同时说出它所在的队列和还没进入的队列，例如：`pending，还没进 running`。


In [6]:
llm.pending_requests = [(PROMPT, params)]
messages = llm.offline_receive_msg()
msg = messages[0]
llm._process_one_msg(msg)
print("message type:", type(msg).__name__)
print("pending uids:", [p.uid for p in llm.prefill_manager.pending_list])
print("decode running uids:", [r.uid for r in llm.decode_manager.running_reqs])
print("no model forward has happened yet")

gate(2, "第 2 关：请求此刻在哪？请包含 pending 和 running（或 decode）的判断：", ["pending", "未", "not", "还没"])


message type: UserMsg
pending uids: [0]
decode running uids: []
no model forward has happened yet


第 2 关：请求此刻在哪？请包含 pending 和 running（或 decode）的判断：
>  pending


第 2 关通过。


## 第 3 关：Prefill batch

Prefill 阶段一次处理 prompt 中尚未缓存的 token。Scheduler 现在要从 pending 请求中挑选一个 batch。

**问题：为什么第一次 forward 是 prefill，而不是 decode？**
提示：第一次需要处理的是 prompt，还是已经生成的单个 token？


In [ ]:
batch = llm.prefill_manager.schedule_next_batch(llm.prefill_budget)
lesson["batch"] = batch
print("batch phase:", batch.phase)
print("batch size:", batch.size)
print("request states:", [req_state(r) for r in batch.reqs])
print("prefill budget:", llm.prefill_budget)

gate(3, "第 3 关：第一次为什么是 prefill？请回答 prompt / 输入 token：", ["prompt", "输入", "input"])


## 第 4 关：把 batch 变成 GPU metadata

Scheduler 不只是“选请求”，还要把请求整理成 attention backend 和 kernel 能使用的地址信息。

下面会生成：

- `positions`：这些 token 在序列中的位置；
- `input_tuple`：输入 token 如何映射到 token pool；
- `write_tuple`：输出 token 写回哪个请求/位置；
- `out_loc`：本轮 KV 写入的位置。

**问题：这些 metadata 的主要目的是什么？**
不是生成文本，而是帮助 GPU 正确找到输入、KV cache 和输出位置。


In [ ]:
forward_input = llm._prepare_batch(batch)
lesson["forward_input"] = forward_input
print("phase:", batch.phase)
print("positions shape:", tuple(batch.positions.shape))
print("input tuple shapes:", [tuple(x.shape) for x in forward_input.input_tuple])
print("write tuple shapes:", [tuple(x.shape) for x in forward_input.write_tuple])
print("out_loc shape:", tuple(batch.out_loc.shape))
print("page table row:", batch.reqs[0].table_idx)

gate(4, "第 4 关：metadata 是为了帮助 GPU 找到什么？请回答 metadata / 地址 / KV cache：", ["metadata", "地址", "address", "kv", "cache"])


## 第 5 关：Engine forward 与 sampling

现在才真正进入 Engine：

```text
model.forward() → logits → sampler.sample() → next token
```

**问题：先发生 model forward，还是先发生 sampling？**


In [ ]:
with llm.engine_stream_ctx:
    forward_output = llm._forward(forward_input)
lesson["forward_output"] = forward_output
print("phase:", batch.phase)
print("next token on GPU:", forward_output.next_tokens_gpu)
print("next token on CPU:", forward_output.next_tokens_cpu)
print("next token shape:", tuple(forward_output.next_tokens_gpu.shape))
print("logits are consumed inside Engine before sampling")

gate(5, "第 5 关：顺序是什么？请回答 forward 在 sampling 之前：", ["forward", "先", "before", "模型"])


## 第 6 关：从 Prefill 进入 Decode

先处理第 5 关得到的 token。随后 Scheduler 会把仍可继续生成的请求放进 decode manager。

Decode 每次通常只为每个请求前进一步；请求结束后，Scheduler 才释放 table slot 和 KV cache 尾部。

**问题：后续 token 为什么进入 decode？结束时需要做什么？**
请同时包含 `decode` 和 `回收 / 释放 / free`。


In [ ]:
# 处理 prefill 的 next token；这一步会把请求加入 decode manager（如果仍可生成）。
llm._process_last_data((forward_input, forward_output))
print("after prefill result:", [req_state(r) for r in llm.decode_manager.running_reqs])

step = 0
while llm.decode_manager.runnable:
    step += 1
    decode_batch = llm.decode_manager.schedule_next_batch()
    decode_input = llm._prepare_batch(decode_batch)
    with llm.engine_stream_ctx:
        decode_output = llm._forward(decode_input)
    print(f"decode step {step}:", [req_state(r) for r in decode_batch.reqs])
    llm._process_last_data((decode_input, decode_output))
    if step > MAX_TOKENS + 1:
        raise RuntimeError("decode loop exceeded the expected token budget")

print("decode manager empty:", not llm.decode_manager.runnable)
print("offline output ids:", llm.status_map[0].output_ids)
print("decoded text:", llm.tokenizer.decode(llm.status_map[0].output_ids))

gate(6, "第 6 关：后续阶段和结束动作？请同时包含 decode 与回收/释放/free：", ["decode", "回收", "释放", "free", "release"])


## 完成

你已经追完了一个请求的最小核心路径：

```text
字符串
→ token IDs
→ UserMsg / PendingReq
→ Prefill Batch
→ GPU metadata
→ Engine forward
→ Sampling
→ Decode
→ 结果与资源回收
```

下一次学习可以在这条路径上只打开一个复杂度：先把 `cache_type="naive"` 改成 `radix`，再比较 prefix cache；不要同时打开 TP、overlap scheduling 和 CUDA Graph。


In [ ]:
# 释放本 notebook 创建的 CUDA / distributed 资源。
llm.shutdown()
print("resources released")
